# AxiomForgeAI — GRPO Training

Training loop structured around the classic RL interface:

```
env.reset(qa)   →  start episode, receive question
env.step(action)→  submit solution, receive reward + feedback
env.state       →  inspect episode metadata
env.close()     →  persist curriculum, release resources
```

All scoring, curriculum management, and reward computation are handled inside
`AxiomforgeaiEnvironment`.  The notebook owns model loading, solution generation,
GRPO loss, and optimisation.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
from __future__ import annotations

import argparse, copy, csv, hashlib, json, logging, random, re
import shutil, sys, time, types
from collections import defaultdict
from datetime import datetime
from enum import Enum, auto as _auto
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn.functional as F
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Ensure the repo root is always on sys.path regardless of the kernel's cwd.
_REPO_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

# ── RL Environment  (reset / step / state / close) ───────────────────────────
from server.AxiomForgeAI_environment import AxiomforgeaiEnvironment
from models import AxiomforgeaiAction

# ── Existing utilities from scripts/ and src/ ────────────────────────────────
from scripts.convert_gsm8k_to_sft import parse_gsm8k_answer
from scripts.eval_sft_inference import evaluate_gsm8k
from src.rl.prm_scorer import ProcessRewardScorer
from src.rl.math_environment_curriculum import CurriculumMathEnvironment
from src.rl.unified_accuracy import StepChainExtractor, UnifiedAccuracyCalculator
from src.rl.llm_question_classifier import LLMQuestionClassifier
from src.config.prompts import create_generator_messages
from src.sft.solution_format import extract_final_answer_numeric_str
from src.utils.attn_backend import select_attn_implementation
from src.utils.csv_logger import CSVLogger

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s - %(message)s",
)
logger = logging.getLogger(__name__)

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# Edit these values before running.  Every key matches the corresponding
# CLI flag in scripts/run_grpo_training.py for compatibility.

args = argparse.Namespace(
    # ── Paths ─────────────────────────────────────────────────────────────────
    base_model          = "checkpoints/dual_task_v1",
    output_dir          = "checkpoints/grpo",
    gsm8k_data          = "data/sft/gsm8k_sft.jsonl",
    eval_data_path      = "data/sft/gsm8k_test.jsonl",
    math_data           = None,
    extraction_cache    = "data/extraction_cache.json",
    run_name            = None,                     # auto-set to grpo_<timestamp>

    # ── Training scale ────────────────────────────────────────────────────────
    num_iterations      = 60,
    questions_per_iter  = 20,
    group_size          = 10,   # K solutions per question (GRPO group)
    q_group_size        = 2,    # K_q question candidates for two-phase self-play

    # ── Optimiser ─────────────────────────────────────────────────────────────
    learning_rate       = 5e-6,
    max_grad_norm       = 0.5,
    kl_coef             = 0.06,
    clip_eps            = 0.2,
    warmup_iters        = 8,
    min_lr_ratio        = 0.1,

    # ── Generation ────────────────────────────────────────────────────────────
    max_new_tokens      = 1000,
    temperature         = 0.8,
    overlong_filter     = True,

    # ── Dataset mixing (GSM8K → MATH curriculum ramp) ─────────────────────────
    math_mix_ratio      = 0.30,  # MATH fraction at ramp start
    math_mix_ratio_late = 0.50,  # MATH fraction after ramp
    math_ramp_start     = 18,    # iteration at which MATH mix starts increasing
    math_max_difficulty = 3,
    difficulty_alpha    = 3.5,   # Zipf-style sampling; higher → more hard questions

    # ── Evaluation ────────────────────────────────────────────────────────────
    eval_every          = 5,
    eval_max_samples    = 150,
    eval_max_new_tokens = 1000,
    eval_pass_at_k      = 0,
    skip_initial_eval   = False,

    # ── PRM (Process Reward Model) ────────────────────────────────────────────
    use_prm             = True,
    prm_model           = "Qwen/Qwen2.5-Math-PRM-7B",

    # ── Chain / unified accuracy extractor ───────────────────────────────────
    extractor_model     = "Qwen/Qwen2.5-0.5B-Instruct",

    # ── Checkpointing ─────────────────────────────────────────────────────────
    save_every          = 5,
    keep_last           = 4,

    # ── Self-play phase curriculum ────────────────────────────────────────────
    # Phase 1 (GROUNDED_ONLY): grounded-only until min_warmup iters pass AND
    #   grounded accuracy ≥ selfplay_gt_thresh AND step accuracy ≥ selfplay_step_thresh
    # Phase 2 (SELFPLAY_RAMP): linearly ramp self-play over selfplay_ramp_iters
    # Phase 3 (CONTINUOUS): stable mix; falls back to grounded if quality drops
    self_play_ratio         = 0.70,   # target self-play fraction in Phase 3
    min_warmup              = 12,     # minimum grounded-only iterations before SP
    selfplay_gt_thresh      = 0.65,   # gt_match_rate required to unlock self-play
    selfplay_grounded_thresh= 0.65,   # grounded accuracy required to unlock self-play
    selfplay_step_thresh    = 0.68,   # step-level accuracy threshold
    selfplay_ramp_iters     = 28,     # iterations to ramp from 0 → self_play_ratio
    grounded_floor          = 0.55,   # below this grounded acc → suspend self-play
)

In [ ]:
# ── Run identity + directory layout ──────────────────────────────────────────
run_name = args.run_name or f"grpo_{datetime.now():%Y%m%d_%H%M%S}"
out_dir  = Path(args.output_dir) / run_name
log_dir  = Path("logs") / "grpo" / run_name
out_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

# ── Console mirror (TeeStream) ────────────────────────────────────────────────
class TeeStream:
    """Mirrors every write to a terminal stream into a log file."""
    def __init__(self, primary, secondary):
        self.primary, self.secondary = primary, secondary
    def write(self, data):
        self.primary.write(data); self.secondary.write(data); return len(data)
    def flush(self):
        self.primary.flush(); self.secondary.flush()
    def isatty(self):
        return getattr(self.primary, "isatty", lambda: False)()
    def fileno(self):
        return self.primary.fileno()

console_log_path  = log_dir / "console_output.log"
_console_log_file = console_log_path.open("a", encoding="utf-8", buffering=1)

def _add_file_logging(path: Path) -> logging.FileHandler:
    fh = logging.FileHandler(path, mode="a", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)-8s %(name)s - %(message)s"))
    logging.getLogger().addHandler(fh)
    return fh

_file_handler   = _add_file_logging(console_log_path)
_orig_stdout    = sys.stdout
_orig_stderr    = sys.stderr
sys.stdout      = TeeStream(_orig_stdout, _console_log_file)
sys.stderr      = TeeStream(_orig_stderr, _console_log_file)

# ── Live CSV metrics writer (via CSVLogger) ───────────────────────────────────
# CSVLogger writes key metrics to metrics.csv and full metrics as per-step JSON
# under logs/grpo/<run_name>/detailed_metrics/step_NNNN.json
_csv_logger = CSVLogger(
    project="grpo",
    run_name=run_name,
    log_dir="logs",
    config=vars(args),
    log_detailed=True,
)

def _append_metrics_csv(row: Dict[str, Any], step: Optional[int] = None) -> None:
    """Write one row of metrics via CSVLogger (key metrics → CSV, all → JSON)."""
    _csv_logger.log(row, step=step)

# ── Teardown (atexit + explicit) ──────────────────────────────────────────────
def _teardown() -> None:
    sys.stdout = _orig_stdout
    sys.stderr = _orig_stderr
    logging.getLogger().removeHandler(_file_handler)
    if not getattr(_file_handler.stream, "closed", False): _file_handler.close()
    if not _console_log_file.closed: _console_log_file.close()
    _csv_logger.finish()

import atexit; atexit.register(_teardown)

random.seed(42); np.random.seed(42); torch.manual_seed(42)

logger.info("Run: %s  |  out=%s  |  log=%s", run_name, out_dir, log_dir)

In [ ]:
# ── Device + attention backend ────────────────────────────────────────────────
device   = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
attn_impl = select_attn_implementation()
logger.info("Device: %s | attn: %s", device, attn_impl)
if torch.cuda.is_available():
    _g = torch.cuda.get_device_properties(0)
    logger.info("GPU: %s | %.1f GB | sm_%d%d", _g.name, _g.total_memory/1e9, _g.major, _g.minor)

# ── Policy model ──────────────────────────────────────────────────────────────
logger.info("Loading model from %s ...", args.base_model)
tokenizer = AutoTokenizer.from_pretrained(args.base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Patch missing chat_template (common in SFT adapter checkpoints)
if tokenizer.chat_template is None:
    _base_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
    _meta = Path(args.base_model) / "pipeline_meta.json"
    if _meta.exists():
        _base_name = json.loads(_meta.read_text(encoding="utf-8")).get("base_model", _base_name)
    try:
        _bt = AutoTokenizer.from_pretrained(_base_name, trust_remote_code=True)
        if _bt.chat_template: tokenizer.chat_template = _bt.chat_template
        logger.info("Chat template loaded from %s", _base_name)
    except Exception as _e:
        logger.warning("Could not load chat template: %s", _e)

# Patch missing tensor_parallel shim (PEFT ≤ 0.12)
if "transformers.integrations.tensor_parallel" not in sys.modules:
    sys.modules["transformers.integrations.tensor_parallel"] = types.ModuleType("tensor_parallel")

load_kwargs = dict(
    torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
    device_map={"":device}, trust_remote_code=True, attn_implementation=attn_impl)

model_path = Path(args.base_model)
if (model_path / "adapter_config.json").exists():
    _meta_p = model_path / "pipeline_meta.json"
    _base_w = "Qwen/Qwen2.5-Math-1.5B-Instruct"
    if _meta_p.exists():
        _base_w = json.loads(_meta_p.read_text(encoding="utf-8")).get("base_model", _base_w)
    logger.info("PEFT adapter — loading base %s then merging %s", _base_w, args.base_model)
    _base = AutoModelForCausalLM.from_pretrained(_base_w, **load_kwargs)
    model  = PeftModel.from_pretrained(_base, args.base_model).merge_and_unload().to(device)
else:
    model = AutoModelForCausalLM.from_pretrained(args.base_model, **load_kwargs)

for p in model.parameters(): p.requires_grad_(True)

# Flash-Attn 2 makes gradient checkpointing redundant (same O(T) memory)
if attn_impl != "flash_attention_2":
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    if hasattr(model, "config"): model.config.use_cache = False
    logger.info("Gradient checkpointing enabled.")

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in model.parameters())
logger.info("Parameters: %s / %s trainable (%.1f%%)",
            f"{n_trainable:,}", f"{n_total:,}", 100*n_trainable/max(n_total,1))

# ── Frozen reference policy for KL penalty ────────────────────────────────────
ref_model: Optional[AutoModelForCausalLM] = None
if args.kl_coef > 0.0:
    ref_model = copy.deepcopy(model)
    ref_model.requires_grad_(False).eval()
    logger.info("Reference policy ready (kl_coef=%.4f).", args.kl_coef)

In [ ]:
# ── Load training data ────────────────────────────────────────────────────────
def _load_jsonl_qa(path: str) -> List[Dict[str, str]]:
    """Load {question, gold_final} pairs from a JSONL file."""
    pairs: List[Dict[str, str]] = []
    p = Path(path)
    if not p.exists():
        logger.warning("Data file not found: %s", path); return pairs
    with p.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: rec = json.loads(line)
            except json.JSONDecodeError: continue
            if "question" in rec and "answer" in rec:
                q = rec["question"].strip()
                _, g = parse_gsm8k_answer(str(rec["answer"]))
            elif "messages" in rec:
                q, asst = "", ""
                for msg in rec["messages"]:
                    if msg.get("role") == "user"   and not q:    q    = msg.get("content","").strip()
                    if msg.get("role") == "assistant" and not asst: asst = msg.get("content","")
                if "Problem:" in q: q = q.split("Problem:",1)[1].strip()
                g = (extract_final_answer_numeric_str(asst) or "").strip()
            else:
                continue
            if q and g: pairs.append({"question": q, "gold_final": g})
    logger.info("Loaded %d QA pairs from %s", len(pairs), path)
    return pairs

def _load_math_dataset(
    local_path: Optional[str] = None,
    cache: str = "data/math/math_numeric.jsonl",
    max_diff: int = 3,
) -> List[Dict[str, str]]:
    """Load MATH competition dataset (numeric answers, difficulty ≤ max_diff)."""
    for src in filter(None, [local_path, cache]):
        p = Path(src)
        if p.exists():
            items = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
            if items: logger.info("Loaded %d MATH pairs from %s", len(items), p); return items
    try:
        from datasets import load_dataset
        ds = load_dataset("qwedsacf/competition_math", split="train", trust_remote_code=True)
    except Exception as e:
        logger.warning("MATH download failed (%s) — GSM8K only.", e); return []
    pairs, _box = [], re.compile(r"\\\\boxed\\{([^}]*)\\}")
    for item in ds:
        lvl = item.get("level","Level 5")
        try:
            if int(lvl.split()[-1]) > max_diff: continue
        except (ValueError, IndexError): continue
        m = _box.search(item.get("solution",""))
        if not m: continue
        raw = m.group(1).strip()
        try: num = str(int(raw))
        except ValueError:
            try: v=float(raw); num=str(int(v)) if v==int(v) else f"{v:.4f}"
            except ValueError: continue
        pairs.append({"question": item.get("problem","").strip(), "gold_final": num})
    if pairs:
        Path(cache).parent.mkdir(parents=True,exist_ok=True)
        Path(cache).write_text("\n".join(json.dumps(p) for p in pairs), encoding="utf-8")
        logger.info("Cached %d MATH pairs → %s", len(pairs), cache)
    return pairs

gsm8k_pairs = _load_jsonl_qa(args.gsm8k_data)
if not gsm8k_pairs:
    raise SystemExit(f"No training data at {args.gsm8k_data}")

math_pairs: List[Dict[str, str]] = []
if args.math_mix_ratio > 0:
    math_pairs = _load_math_dataset(args.math_data, max_diff=args.math_max_difficulty)
    if math_pairs:
        logger.info("MATH mix: %.0f%% MATH (%d) + %.0f%% GSM8K (%d)",
                    100*args.math_mix_ratio, len(math_pairs),
                    100*(1-args.math_mix_ratio), len(gsm8k_pairs))

# ── PRM scorer ────────────────────────────────────────────────────────────────
prm: Optional[ProcessRewardScorer] = None
if args.use_prm:
    try:
        prm = ProcessRewardScorer(model_name=args.prm_model, device=device, load_in_4bit=True)
        logger.info("PRM loaded: %s (4-bit)", args.prm_model)
    except Exception as e:
        logger.warning("PRM load failed (%s) — no PRM scoring.", e)

# ── Unified accuracy calculator (step-chain scoring, Phase 2+) ────────────────
_extractor   = StepChainExtractor(model_name=args.extractor_model, device=str(device),
                                   cache_path=args.extraction_cache)
_unified_calc = UnifiedAccuracyCalculator(extractor=_extractor, question_evaluator=None)
logger.info("Warming up step-chain extractor ...")
_extractor.warmup()
logger.info("Extractor ready.")

# ── CurriculumMathEnvironment (full model — generates + scores) ───────────────
math_env = CurriculumMathEnvironment(
    policy_model=model,
    value_model=None,
    tokenizer=tokenizer,
    reference_questions=[p["question"] for p in gsm8k_pairs],
    grounded_qa_pairs=gsm8k_pairs,
    prm_scorer=prm,
    max_solution_tokens=args.max_new_tokens,
    device=device,
    unified_accuracy_calc=_unified_calc,
)
_unified_calc.question_evaluator = math_env.question_evaluator

# LLM-backed question classifier (uses the already-loaded policy)
_llm_cls = LLMQuestionClassifier(model=model, tokenizer=tokenizer,
                                  device=device, cache_size=10_000)
math_env.question_evaluator.classifier = _llm_cls

# Bootstrap curriculum from structured dataset (NuminaMath / OpenMathInstruct)
_raw = [json.loads(l) for l in Path(args.gsm8k_data).read_text(encoding="utf-8").splitlines() if l.strip()]
if any("skill_id" in r for r in _raw[:20]):
    math_env.curriculum_manager.initialize_from_dataset(_raw)
    logger.info("Curriculum bootstrapped from skill_ids.")
else:
    logger.info("Plain dataset — keyword-classifier bootstrap.")

# ── RL Environment — wraps math_env with reset / step / state / close ─────────
env = AxiomforgeaiEnvironment()
env._math_env = math_env   # inject the training-configured math_env
logger.info("RL environment ready — reset / step / state / close.")

In [ ]:
# ── Optimiser + LR schedule ───────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=args.learning_rate,
    fused=torch.cuda.is_available(),
)

from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
_nw = max(1, args.warmup_iters)
_nt = max(1, args.num_iterations)
_nd = max(1, _nt - _nw)
scheduler = SequentialLR(
    optimizer,
    schedulers=[
        LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=_nw),
        CosineAnnealingLR(optimizer, T_max=_nd, eta_min=args.learning_rate * args.min_lr_ratio),
    ],
    milestones=[_nw],
)
logger.info("LR: %.1e  warmup=%d  cosine=%d  min=%.1e",
            args.learning_rate, _nw, _nd, args.learning_rate * args.min_lr_ratio)

In [ ]:
# ── GRPO utilities ────────────────────────────────────────────────────────────
# These functions live in the notebook because they depend on live model
# objects and are tightly coupled to the GRPO update step.

def _stop_ids(tok: AutoTokenizer) -> Optional[List[int]]:
    ids = []
    if tok.eos_token_id is not None: ids.append(tok.eos_token_id)
    im = tok.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im, int) and im not in ids: ids.append(im)
    return ids or None


@torch.no_grad()
def generate_questions_batched(
    model, tokenizer, instruction: str, K_q: int,
    max_new_tokens: int, temperature: float, device,
) -> Tuple[List[str], List, List, List]:
    """Generate K_q candidate question strings from a curriculum instruction."""
    messages = create_generator_messages(instruction)
    try:    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except: prompt = f"{messages[0]['content']}\n\n{instruction}\n"
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    enc    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    plen   = enc["input_ids"].shape[1]
    out = model.generate(
        input_ids=enc["input_ids"].expand(K_q,-1).contiguous(),
        attention_mask=enc["attention_mask"].expand(K_q,-1).contiguous(),
        max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
        top_p=0.95, pad_token_id=pad_id, eos_token_id=_stop_ids(tokenizer), use_cache=True)
    pad_t = torch.tensor(pad_id, device=device, dtype=out.dtype)
    questions, ids_list, masks_list, olps_list = [], [], [], []
    attn_lp = (out != pad_t); attn_lp[:,:plen] = True
    batch_logits = model(input_ids=out, attention_mask=attn_lp.long(),
                         use_cache=False, return_dict=True).logits
    for i in range(K_q):
        full = out[i]; resp = full[plen:]
        mask = torch.zeros(full.shape[0], dtype=torch.bool, device=device)
        mask[plen:] = resp != pad_t
        questions.append(tokenizer.decode(resp, skip_special_tokens=True).strip())
        ids_list.append(full); masks_list.append(mask)
        sl = batch_logits[i,:-1]; lb = full[1:]; sm = mask[1:]
        lp = F.log_softmax(sl,dim=-1)[torch.arange(sl.size(0),device=device), lb]
        resp_lp = lp[sm]
        olps_list.append(resp_lp.sum().detach() if resp_lp.numel()>0 else torch.tensor(0.,device=device))
    return questions, ids_list, masks_list, olps_list


def generate_solutions_batched(
    model, tokenizer, prompt: str, K: int,
    max_new_tokens: int, temperature: float, device,
) -> Tuple[List[str], List, List, List]:
    """Generate K solution strings and their per-sequence log-probs."""
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    enc    = tokenizer(prompt, return_tensors="pt", padding=False,
                       truncation=True, max_length=1024).to(device)
    plen   = enc["input_ids"].shape[1]
    model.eval()
    with torch.no_grad():
        out = model.generate(
            input_ids=enc["input_ids"].expand(K,-1).contiguous(),
            attention_mask=enc["attention_mask"].expand(K,-1).contiguous(),
            max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
            top_p=0.9, pad_token_id=pad_id, eos_token_id=_stop_ids(tokenizer), use_cache=True)
    pad_t = torch.tensor(pad_id, device=device, dtype=out.dtype)
    solutions, ids_list, masks_list, olps_list = [], [], [], []
    with torch.no_grad():
        attn_lp = (out != pad_t); attn_lp[:,:plen] = True
        batch_logits = model(input_ids=out, attention_mask=attn_lp.long(),
                             use_cache=False, return_dict=True).logits
    for i in range(K):
        full = out[i]; resp = full[plen:]
        mask = torch.zeros(full.shape[0], dtype=torch.bool, device=device)
        mask[plen:] = resp != pad_t
        solutions.append(tokenizer.decode(resp, skip_special_tokens=True))
        ids_list.append(full); masks_list.append(mask)
        sl = batch_logits[i,:-1]; lb = full[1:]; sm = mask[1:]
        lp = F.log_softmax(sl,dim=-1)[torch.arange(sl.size(0),device=device), lb]
        resp_lp = lp[sm]
        olps_list.append(resp_lp.sum().detach() if resp_lp.numel()>0 else torch.tensor(0.,device=device))
    return solutions, ids_list, masks_list, olps_list


def compute_sequence_log_prob(model, input_ids, response_mask) -> torch.Tensor:
    """Forward pass → sum of log-probs over the response tokens."""
    logits = model(input_ids=input_ids.unsqueeze(0), use_cache=False, return_dict=True).logits[0]
    lp = F.log_softmax(logits[:-1], dim=-1)
    token_lp = lp[torch.arange(lp.size(0), device=lp.device), input_ids[1:]]
    resp = token_lp[response_mask[1:]]
    return resp.sum() if resp.numel() > 0 else torch.tensor(0., requires_grad=True, device=input_ids.device)


def grpo_loss_for_group(
    model, ids_list, masks_list, rewards: List[float], old_lps,
    clip_eps: float = 0.2, kl_coef: float = 0.0, ref_model=None, eps: float = 1e-8,
) -> Optional[torch.Tensor]:
    """GRPO policy loss for one question group (K solutions)."""
    r = np.array(rewards, dtype=np.float32)
    if r.std() < eps: return None
    advantages = np.clip((r - r.mean()) / (r.std() + eps), -5., 5.)
    dev = next(model.parameters()).device
    loss = torch.tensor(0., device=dev); n = 0
    model.train()
    for ids, mask, adv, olp in zip(ids_list, masks_list, advantages, old_lps):
        n_resp = int(mask[1:].sum().item())
        if n_resp == 0: continue
        new_lp = compute_sequence_log_prob(model, ids, mask)
        adv_t  = torch.tensor(adv, dtype=new_lp.dtype, device=dev)
        if clip_eps > 0:
            ratio = torch.exp(new_lp - olp.to(dev).detach())
            li = -torch.min(ratio * adv_t, torch.clamp(ratio,1-clip_eps,1+clip_eps) * adv_t) / n_resp
        else:
            li = -(adv_t * new_lp / n_resp)
        if kl_coef > 0 and ref_model is not None:
            with torch.no_grad(): ref_lp = compute_sequence_log_prob(ref_model, ids, mask)
            li = li + kl_coef * (new_lp - ref_lp.to(dev).detach()) / n_resp
        loss = loss + li; n += 1
    return loss / n if n > 0 else None


def compute_self_play_reward(
    question: str, solution: str, topic: str, difficulty: float, math_env,
) -> Tuple[float, float, float, Dict]:
    """Self-play reward via math_env.compute_reward (no gold answer)."""
    result  = math_env.compute_reward(question=question, solution=solution,
                                      target_topic=topic, target_difficulty=difficulty)
    combined = float(result["combined_score"])
    sol_m    = result.get("solution_metrics") or {}
    s_rew    = float(sol_m.get("overall_score", 0.)) if isinstance(sol_m, dict) else 0.
    q_raw    = result.get("question_metrics") or {}
    q_rew    = float(result.get("effective_question_reward", q_raw.get("overall_score", 0.)))
    q_met: Dict = {
        "overall_score":  q_rew,
        "topic_match":    float(q_raw.get("topic_match",       0.)),
        "difficulty_fit": float(q_raw.get("difficulty_score", 0.)),
        "clarity":        float(q_raw.get("clarity",          0.)),
        "novelty":        float(q_raw.get("novelty_combined", 0.)),
        "solvability":    float(q_raw.get("solvability_score",0.)),
        "sp_chain_integrity_score": result.get("sp_chain_integrity_score"),
    }
    return combined, q_rew, s_rew, q_met


def _verify_sp_answer(solutions: List[str], topic: str, difficulty: float) -> bool:
    """Consensus check: majority of K solutions agree on a numeric answer."""
    t = topic.lower().replace(" ","_")
    if t in {"geometry"} or difficulty >= 4.: return False
    answers: List[float] = []
    for sol in solutions:
        m = re.search(r"final answer[:\s]*([^\n]+)", sol, re.I)
        if not m: continue
        raw = m.group(1).strip()
        for fn in (lambda s: float(eval(s, {"__builtins__":{}}, {})),
                   lambda s: float(__import__("sympy").N(__import__("sympy").sympify(s), 15))):
            try: v = fn(raw); answers.append(round(v, 6)); break
            except: pass
    if not answers: return False
    maj = max(set(answers), key=answers.count)
    return answers.count(maj) >= max(1, len(solutions)//2)


def evaluate_policy(
    model, tokenizer, data_path: str, max_samples: int,
    max_new_tokens: int, math_env=None, pass_at_k: int = 4,
) -> Dict[str, Any]:
    """Run evaluation on held-out data; returns combined_score and related metrics."""
    if not Path(data_path).exists(): return {"accuracy": 0., "combined_score": 0., "total": 0}
    model.eval()
    reward_fn = None
    if math_env is not None:
        import logging as _lm
        _ml = _lm.getLogger("src.rl.math_environment_curriculum")
        _pl = _lm.getLogger("src.rl.prm_scorer")
        def reward_fn(q, s, g):
            _ml.setLevel(_lm.WARNING); _pl.setLevel(_lm.WARNING)
            try:    return math_env.compute_grounded_reward(q, s, g)
            finally: _ml.setLevel(_lm.INFO); _pl.setLevel(_lm.INFO)
    stem = Path(data_path).stem.lower()
    ds_name = "AQuA-RAT" if "aqua" in stem else "MATH" if "math" in stem else "GSM8K"
    results = evaluate_gsm8k(model=model, tokenizer=tokenizer, data_path=data_path,
                              max_samples=max_samples, max_new_tokens=max_new_tokens,
                              reward_fn=reward_fn, pass_at_k=pass_at_k, dataset_name=ds_name)
    model.train()
    return results


# ── Difficulty-adaptive question sampling ─────────────────────────────────────
_q_wins:     Dict[str, int] = defaultdict(int)
_q_attempts: Dict[str, int] = defaultdict(int)

def _qkey(q: str) -> str:
    return hashlib.md5(q.encode(), usedforsecurity=False).hexdigest()

def _sample_by_difficulty(pool: List[Dict], n: int, alpha: float) -> List[Dict]:
    """Weight questions by how informative they are (win-rate close to 50%)."""
    if alpha <= 0: return random.sample(pool, min(n, len(pool)))
    weights = []
    for qa in pool:
        att = _q_attempts[_qkey(qa["question"])]
        w = 0.75 if att == 0 else max(
            (1. - abs(_q_wins[_qkey(qa["question"])]/att - 0.5)*2.) ** alpha, 0.05)
        weights.append(w)
    tw = sum(weights)
    probs = [w/tw for w in weights]
    return [pool[i] for i in np.random.choice(len(pool), size=min(n,len(pool)), replace=False, p=probs)]

In [ ]:
# ── Optional initial evaluation (Iteration 0 baseline) ───────────────────────
metrics_log: List[Dict] = []
best_combined = best_prm_mean = best_accuracy = 0.

if not args.skip_initial_eval:
    logger.info("=" * 70)
    logger.info("INITIAL EVALUATION (Iteration 0)")
    logger.info("=" * 70)
    _init = evaluate_policy(model, tokenizer, args.eval_data_path,
                             args.eval_max_samples, args.eval_max_new_tokens,
                             math_env=math_env, pass_at_k=args.eval_pass_at_k)
    best_combined = best_accuracy = float(_init.get("combined_score", 0.))
    best_prm_mean = float(_init.get("prm_mean", 0.))
    logger.info("Baseline combined_score=%.4f  correct=%.1f%%",
                best_combined, 100*float(_init.get("correct_rate", 0.)))
    metrics_log.append({"iteration": 0, **_init})

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GRPO Training Loop  —  reset → step → state → close
# ══════════════════════════════════════════════════════════════════════════════

# ── Phase curriculum state ────────────────────────────────────────────────────
class _Phase(Enum):
    GROUNDED_ONLY = _auto()   # grounded only until model is ready
    SELFPLAY_RAMP = _auto()   # ramp self-play ratio up from 0
    CONTINUOUS    = _auto()   # steady-state mixed training

_phase: _Phase           = _Phase.GROUNDED_ONLY
_sp_iters:         int   = 0
_sp_suspended:     bool  = False
_eff_sp:           float = 0.
_use_chain:        bool  = False
_chain_corr:       float = 0.
_extract_rate:     float = 0.
_chain_buf:  List[float] = []
_prm_buf:    List[float] = []
_succ_buf:   List[int]   = []
_CWIN, _CMAX, _SHDW = 50, 200, 4
_shadow_ctr:       int   = 0

for iteration in range(1, args.num_iterations + 1):
    iter_start = time.perf_counter()
    logger.info("=" * 70)
    logger.info("GRPO ITERATION %d / %d  [phase=%s]", iteration, args.num_iterations, _phase.name)
    logger.info("=" * 70)

    # ── Dataset batch (with MATH ramp) ────────────────────────────────────────
    _eff_math = args.math_mix_ratio
    if args.math_mix_ratio_late and iteration > args.math_ramp_start:
        _r = min(1., (iteration - args.math_ramp_start) / 10.)
        _eff_math = args.math_mix_ratio + _r * (args.math_mix_ratio_late - args.math_mix_ratio)
    if math_pairs and _eff_math > 0:
        nm = max(1, round(args.questions_per_iter * _eff_math))
        ng = max(1, args.questions_per_iter - nm)
        batch = (_sample_by_difficulty(math_pairs, nm, args.difficulty_alpha) +
                 _sample_by_difficulty(gsm8k_pairs, ng, args.difficulty_alpha))
        random.shuffle(batch)
    else:
        batch = _sample_by_difficulty(gsm8k_pairs, args.questions_per_iter, args.difficulty_alpha)

    # Temperature annealing: 0.8 → 0.4 over the full run
    _ann = min(1., (iteration-1) / max(1, args.num_iterations-1))
    _temp = args.temperature * (1. - 0.5 * _ann)

    # ── Effective self-play ratio (phase-dependent) ────────────────────────────
    if   _phase == _Phase.GROUNDED_ONLY or _sp_suspended: _eff_sp = 0.
    elif _phase == _Phase.SELFPLAY_RAMP:
        _eff_sp = 1. - max(0.30, 1. - _sp_iters / max(1, args.selfplay_ramp_iters))
    else: _eff_sp = args.self_play_ratio

    _sp_idx = set(random.sample(range(len(batch)), int(round(len(batch)*_eff_sp))))

    # ── Per-iteration metric accumulators ─────────────────────────────────────
    all_r, all_qr = [], []
    gr_r, sp_r    = [], []
    gr_step, gr_lccp, gr_gt = [], [], []
    ch_arith, ch_dep, ch_int, sp_ch = [], [], [], []
    qc = dict(topic=[], diff=[], clarity=[], novelty=[], solvab=[])
    skipped = n_grps = n_sp = q_att = q_val = q_good = 0
    skip0var = 0; total_loss = 0.

    optimizer.zero_grad()

    pbar = tqdm(batch, desc=f"Iter {iteration}", unit="q")
    for _idx, qa in enumerate(pbar):
        is_sp = _idx in _sp_idx

        # ════════════════════════════════════════════════════════════════════
        # RESET  —  start a new episode
        # ════════════════════════════════════════════════════════════════════
        if is_sp:
            # Self-play: curriculum provides the instruction
            instr, topic, diff = env._math_env.sample_instruction()
            if diff >= 4.: skipped += 1; continue
            q_att += 1

            if args.q_group_size > 1:
                # Two-phase SP: generate K_q questions, then K solutions per question
                _qt = min(0.90, _temp + 0.05)
                qcands, qids, qmasks, qolps = generate_questions_batched(
                    model, tokenizer, instr, args.q_group_size, 128, _qt, device)
                vq = [(q,i,m,o) for q,i,m,o in zip(qcands,qids,qmasks,qolps) if len(q.strip())>=10]
                if not vq: skipped += 1; continue
                q_val += 1; n_sp += 1
                qagg: List[float] = []
                for _qt2, _qi, _qm, _qo in vq:
                    sols, sids, smasks, solps = generate_solutions_batched(
                        model, tokenizer, math_env.format_solution_prompt(_qt2),
                        args.group_size, args.max_new_tokens, _temp, device)
                    if args.overlong_filter:
                        vf = [(s,i,m,o) for s,i,m,o in zip(sols,sids,smasks,solps)
                              if int(m.sum())<args.max_new_tokens]
                        if vf: sols,sids,smasks,solps = map(list,zip(*vf))
                        else:  skipped+=1; qagg.append(0.); continue
                    # STEP: score self-play solutions
                    rs: List[float] = []
                    for s in sols:
                        r,qr,_,qm = compute_self_play_reward(_qt2,s,topic,diff,math_env)
                        rs.append(r); all_qr.append(qr)
                        qc["topic"].append(qm["topic_match"]); qc["diff"].append(qm["difficulty_fit"])
                        qc["clarity"].append(qm["clarity"]); qc["novelty"].append(qm["novelty"])
                        qc["solvab"].append(qm["solvability"])
                    all_r.extend(rs); sp_r.extend(rs); qagg.append(float(np.mean(rs)))
                    sl = grpo_loss_for_group(model,sids,smasks,rs,solps,
                                            args.clip_eps,args.kl_coef,ref_model)
                    if sl: sl.backward(); total_loss+=sl.item(); n_grps+=1
                    else:  skipped+=1; skip0var+=1
                # Question GRPO
                ql = grpo_loss_for_group(model,[t[1] for t in vq],[t[2] for t in vq],
                                         qagg,[t[3] for t in vq],args.clip_eps,0.,None)
                if ql: ql.backward()
                if any(r>0.5 for r in qagg): q_good+=1
                pbar.set_postfix(loss=f"{total_loss/max(1,n_grps):.4f}",
                                 sp_r=f"{np.mean(sp_r or [0]):.3f}",skip=skipped)
                continue

            # Single-question self-play
            from src.config.prompts import create_generator_messages as _cgm
            _msgs = _cgm(instr)
            try:    _pr = tokenizer.apply_chat_template(_msgs,tokenize=False,add_generation_prompt=True)
            except: _pr = f"{_msgs[0]['content']}\n\n{instr}\n"
            _enc = tokenizer(_pr,return_tensors="pt",truncation=True,max_length=512).to(device)
            _plen = _enc["input_ids"].shape[1]
            with torch.no_grad():
                _out = model.generate(
                    **_enc, max_new_tokens=128, do_sample=True,
                    temperature=min(0.90,_temp+0.05), top_p=0.95,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                    eos_token_id=_stop_ids(tokenizer), use_cache=True)
            question = tokenizer.decode(_out[0][_plen:], skip_special_tokens=True).strip()
            if len(question.strip()) < 10: skipped+=1; continue
            q_val+=1; n_sp+=1
        else:
            # ── RESET (grounded): inject difficulty-sampled QA pair ────────────
            obs = env.reset(qa=qa)        # state: question from dataset
            question = obs.question       # the grounded math question
            topic, diff = "grounded", 0.5

        # ════════════════════════════════════════════════════════════════════
        # GENERATE  —  policy produces K candidate solutions
        # ════════════════════════════════════════════════════════════════════
        solutions, ids_list, masks_list, lps_list = generate_solutions_batched(
            model, tokenizer, math_env.format_solution_prompt(question),
            args.group_size, args.max_new_tokens, _temp, device)
        if args.overlong_filter:
            vf = [(s,i,m,o) for s,i,m,o in zip(solutions,ids_list,masks_list,lps_list)
                  if int(m.sum()) < args.max_new_tokens]
            if vf: solutions,ids_list,masks_list,lps_list = map(list, zip(*vf))
            else:  skipped+=1; continue

        # ════════════════════════════════════════════════════════════════════
        # STEP  —  score each solution with the RL environment
        # ════════════════════════════════════════════════════════════════════
        rewards: List[float] = []
        _sp_qr: List[float]  = []
        for sol in solutions:
            if is_sp:
                # Self-play: env.compute_reward (no gold answer)
                r, qr, _, qm = compute_self_play_reward(question, sol, topic, diff, math_env)
                _sp_qr.append(qr); all_qr.append(qr)
                qc["topic"].append(qm["topic_match"]); qc["diff"].append(qm["difficulty_fit"])
                qc["clarity"].append(qm["clarity"]); qc["novelty"].append(qm["novelty"])
                qc["solvab"].append(qm["solvability"])
                _sc = qm.get("sp_chain_integrity_score")
                if _sc is not None: sp_ch.append(float(_sc))
            else:
                # Grounded: env.step → compute_grounded_reward internally
                step_obs = env.step(AxiomforgeaiAction(solution=sol))
                r = step_obs.reward
                m = step_obs.metadata or {}
                gr_step.append(float(m.get("step_accuracy", 0.)))
                gr_lccp.append(float(m.get("lccp", 0.)))
                gr_gt.append(bool(m.get("gt_match", False)))
                if m.get("chain_arith_score")    is not None: ch_arith.append(float(m["chain_arith_score"]))
                if m.get("chain_dep_score")       is not None: ch_dep.append(float(m["chain_dep_score"]))
                if m.get("chain_integrity_score") is not None: ch_int.append(float(m["chain_integrity_score"]))
                # Shadow chain extraction for Phase 2 calibration
                _shadow_ctr += 1
                if (_phase == _Phase.SELFPLAY_RAMP and not _use_chain
                        and _unified_calc and _shadow_ctr % _SHDW == 0):
                    _pps = 0.60*m.get("prm_final_score",0.) + 0.40*m.get("prm_mean_score",0.)
                    try:
                        _sh = _unified_calc.compute(solution=sol,gold_answer=qa["gold_final"],
                                                    question=question,topic="grounded",phase="grounded")
                        _chain_buf.append(_sh.chain_integrity_score)
                        _prm_buf.append(_pps)
                        _succ_buf.append(1 if _sh.extraction_succeeded else 0)
                    except Exception: _succ_buf.append(0)
            rewards.append(r)

        all_r.extend(rewards)
        if is_sp: sp_r.extend(rewards)
        else:     gr_r.extend(rewards)

        if is_sp:
            if _sp_qr and np.mean(_sp_qr) > 0.5: q_good += 1
            if not _verify_sp_answer(solutions, topic, diff): skipped+=1; continue
        else:
            k = _qkey(question)
            _q_attempts[k] += len(solutions)
            _q_wins[k]     += sum(1 for r in rewards if r > float(np.median(rewards)))

        # Zero-variance guard
        if np.std(rewards) < 0.02:
            skipped+=1; skip0var+=1
            pbar.set_postfix(mean_r=f"{np.mean(rewards):.3f}",skip=skipped,loss="0var"); continue

        # GRPO loss
        g_loss = grpo_loss_for_group(model, ids_list, masks_list, rewards, lps_list,
                                     args.clip_eps, args.kl_coef, ref_model)
        if g_loss is None:
            skipped+=1; skip0var+=1
            pbar.set_postfix(mean_r=f"{np.mean(rewards):.3f}",skip=skipped,loss="skip"); continue

        g_loss.backward()
        total_loss += g_loss.item(); n_grps += 1
        pbar.set_postfix(mean_r=f"{np.mean(rewards):.3f}",
                         loss=f"{g_loss.item():.4f}", skip=skipped)

    # ── Optimiser step ────────────────────────────────────────────────────────
    if n_grps > 0:
        if n_grps > 1:
            for p in model.parameters():
                if p.grad is not None: p.grad.div_(n_grps)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], args.max_grad_norm)
        optimizer.step()
        loss_val = total_loss / n_grps
    else:
        loss_val = 0.
    scheduler.step()

    # ════════════════════════════════════════════════════════════════════════
    # STATE  —  collect iteration metrics + phase transitions
    # ════════════════════════════════════════════════════════════════════════
    _epi_state = env.state   # episode_id + step_count for the last episode
    iter_time  = time.perf_counter() - iter_start
    mean_r  = float(np.mean(all_r))             if all_r  else 0.
    std_r   = float(np.std(all_r))              if all_r  else 0.
    acc_r   = float(np.mean([r>0.5 for r in all_r])) if all_r else 0.
    gr_acc  = float(np.mean([r>0.5 for r in gr_r]))  if gr_r  else 0.
    step_a  = float(np.mean(gr_step)) if gr_step else 0.
    lccp_a  = float(np.mean(gr_lccp)) if gr_lccp else 0.
    mean_qr = float(np.mean(all_qr))  if all_qr  else 0.
    gt_rate = float(sum(gr_gt)/len(gr_gt)) if gr_gt else 0.
    cur_lr  = optimizer.param_groups[0]["lr"]

    # Phase transition logic
    if _phase == _Phase.GROUNDED_ONLY:
        if (gt_rate  >= args.selfplay_gt_thresh
                and gr_acc  >= args.selfplay_grounded_thresh
                and step_a  >= args.selfplay_step_thresh
                and iteration >= args.min_warmup):
            _phase = _Phase.SELFPLAY_RAMP
            logger.info("PHASE → SELFPLAY_RAMP at iter %d (gt=%.2f acc=%.2f step=%.2f)",
                        iteration, gt_rate, gr_acc, step_a)
    elif _phase in (_Phase.SELFPLAY_RAMP, _Phase.CONTINUOUS):
        _sp_iters += 1
        if _phase == _Phase.SELFPLAY_RAMP and _sp_iters >= args.selfplay_ramp_iters:
            _phase = _Phase.CONTINUOUS
            logger.info("PHASE → CONTINUOUS at iter %d", iteration)
        # Chain scoring calibration
        if len(_chain_buf) > _CMAX:
            _chain_buf[:] = _chain_buf[-_CMAX:]
            _prm_buf[:]   = _prm_buf[-_CMAX:]
            _succ_buf[:]  = _succ_buf[-_CMAX:]
        if not _use_chain and len(_chain_buf) >= _CWIN:
            try:
                from scipy.stats import pearsonr
                _r2, _ = pearsonr(_chain_buf[-_CWIN:], _prm_buf[-_CWIN:])
                _chain_corr = float(_r2)
            except Exception: _chain_corr = 0.
            _n = len(_succ_buf[-_CWIN:])
            _extract_rate = sum(_succ_buf[-_CWIN:])/_n if _n else 0.
            if _chain_corr >= 0.70 and _extract_rate >= 0.80:
                _use_chain = True; math_env.use_chain_scoring = True
                logger.info("CHAIN PRIMARY activated iter %d: corr=%.2f rate=%.2f",
                            iteration, _chain_corr, _extract_rate)
        _prev_susp = _sp_suspended
        _sp_suspended = bool(gr_gt) and gt_rate < args.grounded_floor
        if _sp_suspended and not _prev_susp:
            logger.warning("GROUNDED FLOOR: self-play suspended (gt=%.2f)", gt_rate)
        elif not _sp_suspended and _prev_susp:
            logger.info("GROUNDED FLOOR: self-play resumed (gt=%.2f)", gt_rate)

    # ── Logging ───────────────────────────────────────────────────────────────
    logger.info(
        "Iter %d | loss=%.4f | r=%.3f±%.3f | gt=%.1f%% | gr_acc=%.1f%% | "
        "step=%.1f%% | lccp=%.1f%% | phase=%s sp=%.0f%% | "
        "grps=%d skip=%d | lr=%.2e | %.1fs",
        iteration, loss_val, mean_r, std_r,
        100*gt_rate, 100*gr_acc, 100*step_a, 100*lccp_a,
        _phase.name, 100*_eff_sp, n_grps, skipped, cur_lr, iter_time)
    if (n_grps+skipped) > 0 and skip0var/(n_grps+skipped) > 0.30:
        logger.warning("STARVATION: %.0f%% zero-var groups — curriculum %s",
                       100*skip0var/(n_grps+skipped),
                       "too easy" if gr_acc>0.75 else "too hard")

    # ── Evaluation (every eval_every iterations) ───────────────────────────────
    iter_metrics: Dict[str, Any] = {
        "iteration": iteration, "loss": loss_val, "mean_reward": mean_r,
        "std_reward": std_r, "batch_accuracy": acc_r, "grounded_accuracy": gr_acc,
        "gt_match_rate": round(gt_rate,4), "step_accuracy": step_a, "lccp": lccp_a,
        "n_groups": n_grps, "skipped_groups": skipped, "learning_rate": cur_lr,
        "iter_time_s": iter_time, "training_phase": _phase.name,
        "effective_sp_ratio": round(_eff_sp,3), "selfplay_suspended": int(_sp_suspended),
        "chain_prm_corr": round(_chain_corr,3), "chain_scoring_active": int(_use_chain),
        "n_sp_groups": n_sp, "mean_q_reward": round(mean_qr,4),
        "q_gen_valid_rate": round(q_val/q_att if q_att>0 else 0,4),
        "episode_id": _epi_state.episode_id,   # from env.state
        "episode_steps": _epi_state.step_count, # from env.state
    }

    if iteration % args.eval_every == 0:
        logger.info("Evaluating (%d samples) ...", args.eval_max_samples)
        eval_res = evaluate_policy(model, tokenizer, args.eval_data_path,
                                   args.eval_max_samples, args.eval_max_new_tokens,
                                   math_env=math_env, pass_at_k=args.eval_pass_at_k)
        cur_comb = float(eval_res.get("combined_score", best_combined))
        logger.info("Eval combined=%.4f  correct=%.1f%%  best=%.4f",
                    cur_comb, 100*float(eval_res.get("correct_rate",0.)), best_combined)
        if cur_comb > best_combined + 1e-4:
            best_combined = cur_comb
            best_prm_mean = max(best_prm_mean, float(eval_res.get("prm_mean",0.)))
            model.save_pretrained(str(out_dir/"best_policy"))
            tokenizer.save_pretrained(str(out_dir/"best_policy"))
            logger.info("New best → %s", out_dir/"best_policy")
        iter_metrics.update(eval_res)

    # ── Checkpoint ────────────────────────────────────────────────────────────
    if iteration == args.num_iterations or (args.save_every>0 and iteration%args.save_every==0):
        ck = out_dir / f"iter_{iteration:04d}"
        ck.mkdir(exist_ok=True)
        model.save_pretrained(str(ck)); tokenizer.save_pretrained(str(ck))
        if args.keep_last and args.keep_last > 0:
            old = sorted(p for p in out_dir.iterdir() if p.is_dir() and p.name.startswith("iter_"))
            for o in old[:-args.keep_last]:
                try: shutil.rmtree(o); logger.info("Pruned: %s", o.name)
                except OSError as e: logger.warning("Could not prune %s: %s", o.name, e)

    # ── Write metrics ─────────────────────────────────────────────────────────
    metrics_log.append(iter_metrics)
    (out_dir/"metrics.jsonl").write_text(
        "\n".join(json.dumps(m) for m in metrics_log), encoding="utf-8")
    _append_metrics_csv({
        "iteration":    iter_metrics["iteration"],
        "timestamp":    datetime.now().isoformat(timespec="seconds"),
        "loss":         iter_metrics.get("loss",0.),
        "mean_reward":  iter_metrics.get("mean_reward",0.),
        "batch_acc":    iter_metrics.get("batch_accuracy",0.),
        "grounded_acc": iter_metrics.get("grounded_accuracy",0.),
        "gt_match":     iter_metrics.get("gt_match_rate",0.),
        "step_acc":     iter_metrics.get("step_accuracy",0.),
        "lccp":         iter_metrics.get("lccp",0.),
        "n_groups":     iter_metrics.get("n_groups",0),
        "skipped":      iter_metrics.get("skipped_groups",0),
        "sp_ratio":     iter_metrics.get("effective_sp_ratio",0.),
        "phase":        iter_metrics.get("training_phase",""),
        "lr":           iter_metrics.get("learning_rate",0.),
        "iter_s":       iter_metrics.get("iter_time_s",0.),
        "eval_combined":iter_metrics.get("combined_score","") if "combined_score" in iter_metrics else "",
        "eval_correct": iter_metrics.get("correct_rate","")   if "combined_score" in iter_metrics else "",
        "eval_prm":     iter_metrics.get("prm_mean","")        if "combined_score" in iter_metrics else "",
    }, step=iter_metrics["iteration"])


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CLOSE  —  persist curriculum state and finalise run
# ════════════════════════════════════════════════════════════════════════════
env.close()   # saves CurriculumManager state to checkpoints/curriculum/
_teardown()   # restore stdout/stderr, flush CSV and log files

logger.info("=" * 70)
logger.info("GRPO training complete.")
logger.info("Best combined score : %.4f", best_combined)
logger.info("Best PRM mean       : %.3f", best_prm_mean)
logger.info("Checkpoints         : %s",   out_dir)
logger.info("Logs                : %s",   log_dir)
logger.info("=" * 70)

summary = {
    "run_name":      run_name,
    "best_combined": best_combined,
    "best_prm_mean": best_prm_mean,
    "total_iters":   args.num_iterations,
    "checkpoints":   str(out_dir),
    "log_dir":       str(log_dir),
    "metrics_csv":   str(_csv_logger.metrics_file),
    "metrics_json":  str(_csv_logger.log_path / "detailed_metrics"),
}
_csv_logger.save_summary(summary)
logger.info("Summary → %s", _csv_logger.log_path / "summary.json")

# Auto-generate training plots if matplotlib is available
_jsonl = out_dir / "metrics.jsonl"
if _jsonl.exists():
    try:
        from scripts.plot_grpo_run import generate_plots
        _pdir = generate_plots(_jsonl)
        logger.info("Plots → %s", _pdir)
    except Exception as _pe:
        logger.warning("Plot generation skipped (%s). Run manually: "
                       "python scripts/plot_grpo_run.py %s", _pe, _jsonl)